In [1]:
import subprocess, sys

pkgs = {'xgboost': 'xgboost>=2.0', 'lightgbm': 'lightgbm>=4.0'}
restart_needed = False
for mod, pkg in pkgs.items():
    try:
        __import__(mod)
        print(f'{mod} instalado ✅ — continuar')
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        print(f'{mod} instalado ✅ — REINICIAR KERNEL antes de continuar')
        restart_needed = True
if not restart_needed:
    print('Listo — continuar con celda 0')


xgboost instalado ✅ — continuar
lightgbm instalado ✅ — continuar
Listo — continuar con celda 0


In [2]:
import os, sys, time, pickle, warnings, gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (cohen_kappa_score, f1_score,
                             balanced_accuracy_score, confusion_matrix)
from itertools import product
import mlflow
warnings.filterwarnings('ignore')

SAVE_DIR = '/srv/work'
IN_FEAT  = f'{SAVE_DIR}/te_features_v5.parquet'
OUT_DIR  = f'{SAVE_DIR}/models'
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
HORIZON      = 2
LABEL_COLS   = {0: 'Normal', 1: 'Baja', 2: 'Media', 3: 'Alta'}
ROUTE_KEYS   = ['source_agent', 'cloud_provider', 'region']
ID_COLS      = ROUTE_KEYS + ['window']
TARGET_COL   = 'label_ventana'
SPLIT_COL    = 'split'

UMBRALES = {
    'qwk':        {'floor': 0.65, 'target': 0.72, 'stretch': 0.80},
    'recall_alta':{'floor': 0.55, 'target': 0.75, 'stretch': 0.85},
}

try:
    mlflow.set_tracking_uri('http://localhost:6000')
    mlflow.set_experiment('latam2cloud')
    MLFLOW_OK = True; print('MLflow OK — http://localhost:6000')
except Exception:
    MLFLOW_OK = False; print('MLflow no disponible — continúa sin tracking')

print(f'XGBoost {xgb.__version__}  |  LightGBM {lgb.__version__}')
print(f'Input : {IN_FEAT}')
print(f'Output: {OUT_DIR}')


MLflow OK — http://localhost:6000
XGBoost 3.2.0  |  LightGBM 4.7.0
Input : /srv/work/te_features_v5.parquet
Output: /srv/work/models


In [3]:
import gc

print('Cargando te_features_v5.parquet...')
df = pq.read_table(IN_FEAT).to_pandas()
print(f'  Shape: {df.shape}')

df['y'] = df.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
df = df.dropna(subset=['y'])
df['y'] = df['y'].astype(int)
print(f'  Post target-shift: {df.shape}')

feat_cols_v5 = [c for c in df.columns
                if c not in ID_COLS + [TARGET_COL, SPLIT_COL, 'y']
                and not c.startswith('n_samp')]
print(f'  Features: {len(feat_cols_v5)}')

train_mask = df['split'] == 'train'
val_mask   = df['split'] == 'val'
y_train = df.loc[train_mask, 'y'].values.astype(int)
y_val   = df.loc[val_mask,   'y'].values.astype(int)

ratio_cols_rf = [c for c in feat_cols_v5 if 'ratio' in c]
other_nan_rf  = [c for c in feat_cols_v5 if c not in ratio_cols_rf]

print('\nExtrayendo X_train_rf (float32, imputation RF)...')
X_train_rf = df.loc[train_mask, feat_cols_v5].values.astype('float32')
ratio_idx  = [i for i, c in enumerate(feat_cols_v5) if 'ratio' in c]
other_idx  = [i for i, c in enumerate(feat_cols_v5) if 'ratio' not in c]
for idx in ratio_idx:
    m = np.isnan(X_train_rf[:, idx]); 
    if m.any(): X_train_rf[m, idx] = 1.0
for idx in other_idx:
    m = np.isnan(X_train_rf[:, idx])
    if m.any(): X_train_rf[m, idx] = 0.0
gc.collect()

print('Extrayendo X_val_rf (float32)...')
X_val_rf = df.loc[val_mask, feat_cols_v5].values.astype('float32')
for idx in ratio_idx:
    m = np.isnan(X_val_rf[:, idx])
    if m.any(): X_val_rf[m, idx] = 1.0
for idx in other_idx:
    m = np.isnan(X_val_rf[:, idx])
    if m.any(): X_val_rf[m, idx] = 0.0
gc.collect()

del df; gc.collect()

with open(f'{OUT_DIR}/scaler_v5.pkl', 'rb') as f:
    scaler = pickle.load(f)
X_train_sc = scaler.transform(X_train_rf)
del X_train_rf; gc.collect()
X_val_sc   = scaler.transform(X_val_rf)
del X_val_rf; gc.collect()

class_counts = np.bincount(y_train, minlength=4)
print(f'\nDistribución train:')
for k, name in LABEL_COLS.items():
    print(f'  {name:<8}: {class_counts[k]:>9,}  ({100*class_counts[k]/len(y_train):.1f}%)')
print(f'  X_train_sc: {X_train_sc.shape}  X_val_sc: {X_val_sc.shape}')
print('Celda 1 OK')


Cargando te_features_v5.parquet...
  Shape: (3763917, 109)
  Post target-shift: (3761847, 110)
  Features: 100

Extrayendo X_train_rf (float32, imputation RF)...
Extrayendo X_val_rf (float32)...

Distribución train:
  Normal  : 1,899,347  (89.3%)
  Baja    :   153,319  (7.2%)
  Media   :    36,421  (1.7%)
  Alta    :    36,691  (1.7%)
  X_train_sc: (2125778, 100)  X_val_sc: (644690, 100)
Celda 1 OK


In [4]:
def evaluar(nombre, y_true, y_prob):
    y_pred = np.argmax(y_prob, axis=1)
    qwk    = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    mf1    = f1_score(y_true, y_pred, average='macro', zero_division=0)
    bac    = balanced_accuracy_score(y_true, y_pred)
    cm     = confusion_matrix(y_true, y_pred, labels=[0,1,2,3])
    n_alta = cm[3].sum()
    rec_a  = cm[3,3] / n_alta if n_alta > 0 else 0.0
    prec_a = cm[3,3] / cm[:,3].sum() if cm[:,3].sum() > 0 else 0.0
    mae_o  = np.mean(np.abs(y_true - y_pred))
    cand   = qwk >= UMBRALES['qwk']['floor'] and rec_a >= UMBRALES['recall_alta']['floor']
    sep = '=' * 110
    print(f'\n{sep}')
    print(f'  {nombre} — Validacion')
    print(f'  QWK={qwk:.4f}  MacroF1={mf1:.4f}  BalAcc={bac:.4f}  MAEord={mae_o:.4f}'
          f'  RecAlta={rec_a:.4f}  PrecAlta={prec_a:.4f}')
    if cand:
        print('  ✅ CANDIDATO H2  (QWK≥0.65=True  RecAlta≥0.55=True)')
    else:
        print(f'  ❌ No candidato  (QWK≥0.65={qwk>=0.65}  RecAlta≥0.55={rec_a>=0.55})')
    print(f'{sep}')
    print(f'  Confusion matrix (filas=real, cols=pred):')
    print(f'  {"":>12} {"Normal":>8} {"Baja":>8} {"Media":>8} {"Alta":>8}')
    for k in range(4):
        print(f'  {LABEL_COLS[k]:<12} ' + '  '.join(f'{cm[k,j]:>8,}' for j in range(4)))
    return dict(qwk=qwk, macro_f1=mf1, bal_acc=bac, recall_alta=rec_a,
                prec_alta=prec_a, mae_ord=mae_o, cm=cm, candidato=cand)

refs = {
    'rf_v5':          dict(qwk=0.6420, recall_alta=0.6726, macro_f1=0.6560, bal_acc=0.6720),
    'xgb_v5':         dict(qwk=0.6232, recall_alta=0.6648, macro_f1=0.6511, bal_acc=0.6985),
    'lgb_v5':         dict(qwk=0.6300, recall_alta=0.6824, macro_f1=0.6533, bal_acc=0.7025),
    'ens_rfxgb_v6':   dict(qwk=0.6413, recall_alta=0.5605, macro_f1=0.6563, bal_acc=0.6164),
    'xgb_v7':         dict(qwk=0.6227, recall_alta=0.6445, macro_f1=0.6503, bal_acc=0.6908),
    'lgb_v7':         dict(qwk=0.6299, recall_alta=0.6300, macro_f1=0.6543, bal_acc=0.6832),
    'ens_grid_v7':    dict(qwk=0.6453, recall_alta=0.6685, macro_f1=0.6605, bal_acc=0.6779),
    'ens_thresh_v8':  dict(qwk=0.6507, recall_alta=0.4940, macro_f1=0.6488, bal_acc=0.6670),
}
best_prev = max(refs.items(), key=lambda x: x[1]['qwk'])
print(f'Mejor QWK previo (val): {best_prev[0]} = {best_prev[1]["qwk"]:.4f}')
print(f'Mejor RecAlta+QWK: ens_grid_v7 — QWK=0.6453 RecAlta=0.6685')
print('Función evaluar lista.')


Mejor QWK previo (val): ens_thresh_v8 = 0.6507
Mejor RecAlta+QWK: ens_grid_v7 — QWK=0.6453 RecAlta=0.6685
Función evaluar lista.


In [5]:
print('Cargando probabilidades XGB/LGB v7...')
proba_xgb_v7 = np.load(f'{OUT_DIR}/xgb_v7_val_proba.npy').astype('float32')
proba_lgb_v7 = np.load(f'{OUT_DIR}/lgb_v7_val_proba.npy').astype('float32')
print(f'  proba_xgb_v7: {proba_xgb_v7.shape}')
print(f'  proba_lgb_v7: {proba_lgb_v7.shape}')
print('Celda 3 OK')


Cargando probabilidades XGB/LGB v7...
  proba_xgb_v7: (644690, 4)
  proba_lgb_v7: (644690, 4)
Celda 3 OK


In [ ]:

print('D37 — Entrenando RF v5.1 (n_estimators=300, class_weight=balanced_subsample)...')
print('  Paciencia: ~60-90 min para 2.1M muestras × 100 features')

t0 = time.time()
rf_v51 = RandomForestClassifier(
    n_estimators      = 300,
    max_features      = 'sqrt',
    min_samples_leaf  = 5,
    n_jobs            = 4,
    class_weight      = 'balanced_subsample',
    random_state      = RANDOM_STATE,
    verbose           = 0,
)
rf_v51.fit(X_train_sc, y_train)
elapsed = time.time() - t0
print(f'  Entrenado en {elapsed/60:.1f} min')

print('  Prediciendo en val (chunks 50k)...')
CHUNK = 50_000
n_val = X_val_sc.shape[0]
proba_rf_v51 = np.zeros((n_val, 4), dtype='float32')
t1 = time.time()
for i in range(0, n_val, CHUNK):
    proba_rf_v51[i:i+CHUNK] = rf_v51.predict_proba(X_val_sc[i:i+CHUNK])
print(f'  Predicción val: {time.time()-t1:.1f}s')

np.save(f'{OUT_DIR}/rf_v51_val_proba.npy', proba_rf_v51)
with open(f'{OUT_DIR}/rf_v51_model.pkl', 'wb') as f:
    pickle.dump(rf_v51, f)
print(f'  Guardado: rf_v51_model.pkl + rf_v51_val_proba.npy')

res_rf_v51 = evaluar('RF v5.1 (D37 — 300 estimadores, balanced_subsample)', y_val, proba_rf_v51)

print(f'\n  Δ QWK vs RF v5: {res_rf_v51["qwk"]-0.6420:+.4f}')
print(f'  Δ RecAlta vs RF v5: {res_rf_v51["recall_alta"]-0.6726:+.4f}')

if MLFLOW_OK:
    with mlflow.start_run(run_name='rf_v51'):
        mlflow.log_params({'n_estimators': 300, 'class_weight': 'balanced_subsample',
                           'min_samples_leaf': 5, 'max_features': 'sqrt'})
        mlflow.log_metrics({'qwk_val': res_rf_v51['qwk'],
                            'recall_alta_val': res_rf_v51['recall_alta'],
                            'macro_f1_val': res_rf_v51['macro_f1']})

del rf_v51; gc.collect()
print('Celda 4 OK')


D37 — Entrenando RF v5.1 (n_estimators=300, class_weight=balanced_subsample)...
  Paciencia: ~60-90 min para 2.1M muestras × 100 features


In [7]:
import numpy as np, pickle, gc

proba_rf_v51 = np.load(f'{OUT_DIR}/rf_v51_val_proba.npy').astype('float32')
print(f'  rf_v51_val_proba.npy cargado: {proba_rf_v51.shape}')

try:
    with open(f'{OUT_DIR}/rf_v51_model.pkl', 'wb') as f:
        pickle.dump(rf_v51, f)
    print(f'  rf_v51_model.pkl guardado')
except NameError:
    print('  rf_v51 no en memoria — pkl no guardado (OK, proba ya disponible)')

res_rf_v51 = evaluar('RF v5.1 (D37 — 300 estimadores, balanced_subsample)', y_val, proba_rf_v51)
print(f'\n  Δ QWK vs RF v5:    {res_rf_v51["qwk"]-0.6420:+.4f}')
print(f'  Δ RecAlta vs RF v5: {res_rf_v51["recall_alta"]-0.6726:+.4f}')

if MLFLOW_OK:
    with mlflow.start_run(run_name='rf_v51'):
        mlflow.log_params({'n_estimators': 300, 'class_weight': 'balanced_subsample',
                           'min_samples_leaf': 5, 'max_features': 'sqrt'})
        mlflow.log_metrics({'qwk_val': res_rf_v51['qwk'],
                            'recall_alta_val': res_rf_v51['recall_alta'],
                            'macro_f1_val': res_rf_v51['macro_f1']})

gc.collect()
print('Celda 4 OK')


  rf_v51_val_proba.npy cargado: (644690, 4)
  rf_v51_model.pkl guardado

  RF v5.1 (D37 — 300 estimadores, balanced_subsample) — Validacion
  QWK=0.6503  MacroF1=0.6574  BalAcc=0.6572  MAEord=0.1508  RecAlta=0.6526  PrecAlta=0.5616
  ✅ CANDIDATO H2  (QWK≥0.65=True  RecAlta≥0.55=True)
  Confusion matrix (filas=real, cols=pred):
                 Normal     Baja    Media     Alta
  Normal        525,773    14,755     3,900     6,763
  Baja           17,824    37,730     1,713     1,378
  Media           5,119     1,963     5,597     2,090
  Alta            5,110       581     1,286    13,108

  Δ QWK vs RF v5:    +0.0083
  Δ RecAlta vs RF v5: -0.0200
🏃 View run rf_v51 at: http://localhost:6000/#/experiments/681062908553326840/runs/a0af518138b74c7d9d2737324a2e7f2c
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840
Celda 4 OK


In [8]:

print('D38 — Grid search pesos ensemble RF_v5.1 + XGB_v7 + LGB_v7')
print('  Espacio: w_RF ∈ [0.2..0.9 paso 0.05], XGB+LGB reparten el resto')

best_qwk  = -1
best_cand = False
best_cfg  = None
best_res  = None
top10     = []

w_rf_range = np.arange(0.20, 0.91, 0.05)

for w_rf in w_rf_range:
    rest = 1.0 - w_rf
    for ratio_xgb in np.arange(0.0, 1.01, 0.1):
        w_xgb = round(rest * ratio_xgb, 4)
        w_lgb = round(rest - w_xgb, 4)
        if w_lgb < 0: continue

        p = (w_rf * proba_rf_v51 + w_xgb * proba_xgb_v7 + w_lgb * proba_lgb_v7)
        y_pred = np.argmax(p, axis=1)
        qwk = cohen_kappa_score(y_val, y_pred, weights='quadratic')
        cm  = confusion_matrix(y_val, y_pred, labels=[0,1,2,3])
        rec_a = cm[3,3] / cm[3].sum() if cm[3].sum() > 0 else 0.0
        cand  = qwk >= 0.65 and rec_a >= 0.55
        top10.append((w_rf, w_xgb, w_lgb, qwk, rec_a, cand))
        if qwk > best_qwk or (qwk == best_qwk and cand and not best_cand):
            best_qwk = qwk; best_cand = cand
            best_cfg = (w_rf, w_xgb, w_lgb)

top10 = sorted(top10, key=lambda x: -x[3])[:10]

W_RF, W_XGB, W_LGB = best_cfg
print(f'\nMejor combinación encontrada:')
print(f'  w_RF={W_RF:.4f}  w_XGB={W_XGB:.4f}  w_LGB={W_LGB:.4f}')
print(f'  QWK={best_qwk:.4f}  Candidato={best_cand}')
print(f'\nTop 10 por QWK:')
print(f'  {"w_rf":>6} {"w_xgb":>6} {"w_lgb":>6} {"qwk":>7} {"rec_alta":>9}  cand')
for row in top10:
    print(f'  {row[0]:>6.4f} {row[1]:>6.4f} {row[2]:>6.4f} {row[3]:>7.4f} {row[4]:>9.4f} {row[5]}')

ens_v9_proba = (W_RF * proba_rf_v51 + W_XGB * proba_xgb_v7 + W_LGB * proba_lgb_v7).astype('float32')
res_ens_v9 = evaluar(
    f'Ensemble v9 (w_RF={W_RF} w_XGB={W_XGB} w_LGB={W_LGB}) — D37+D38',
    y_val, ens_v9_proba
)

if MLFLOW_OK:
    with mlflow.start_run(run_name='ens_grid_v9'):
        mlflow.log_params({'w_rf': W_RF, 'w_xgb': W_XGB, 'w_lgb': W_LGB})
        mlflow.log_metrics({'qwk_val': res_ens_v9['qwk'],
                            'recall_alta_val': res_ens_v9['recall_alta'],
                            'macro_f1_val': res_ens_v9['macro_f1']})

np.save(f'{OUT_DIR}/ens_v9_val_proba.npy', ens_v9_proba)
print('Celda 5 OK')


D38 — Grid search pesos ensemble RF_v5.1 + XGB_v7 + LGB_v7
  Espacio: w_RF ∈ [0.2..0.9 paso 0.05], XGB+LGB reparten el resto

Mejor combinación encontrada:
  w_RF=0.8000  w_XGB=0.0000  w_LGB=0.2000
  QWK=0.6529  Candidato=True

Top 10 por QWK:
    w_rf  w_xgb  w_lgb     qwk  rec_alta  cand
  0.8000 0.0000 0.2000  0.6529    0.6509 True
  0.8000 0.0400 0.1600  0.6527    0.6520 True
  0.8000 0.0200 0.1800  0.6527    0.6513 True
  0.8000 0.0600 0.1400  0.6527    0.6520 True
  0.8500 0.0000 0.1500  0.6526    0.6523 True
  0.8500 0.0150 0.1350  0.6525    0.6524 True
  0.8500 0.0600 0.0900  0.6525    0.6537 True
  0.8500 0.0450 0.1050  0.6525    0.6531 True
  0.8000 0.0800 0.1200  0.6525    0.6525 True
  0.8500 0.0750 0.0750  0.6525    0.6541 True

  Ensemble v9 (w_RF=0.7999999999999998 w_XGB=0.0 w_LGB=0.2) — D37+D38 — Validacion
  QWK=0.6529  MacroF1=0.6631  BalAcc=0.6665  MAEord=0.1504  RecAlta=0.6509  PrecAlta=0.5663
  ✅ CANDIDATO H2  (QWK≥0.65=True  RecAlta≥0.55=True)
  Confusion matrix (

In [9]:
modelos_v9 = {
    f'rf_v51 (D37-300+balanced_sub)': res_rf_v51,
    f'ens_grid_v9 (D37+D38)':         res_ens_v9,
}

sep = '=' * 120
print(f'\n{sep}')
print(f'  TABLA COMPARATIVA VALIDACION — v5 / v6 / v7 / v8 / v9')
print(f'{sep}')
hdr = f'  {"Modelo":<49} {"QWK":>8} {"RecAlta":>9} {"MacroF1":>9} {"BalAcc":>8}  H2?'
print(hdr); print(f'  {"-"*118}')
for nombre, r in refs.items():
    cand = r['qwk'] >= 0.65 and r['recall_alta'] >= 0.55
    print(f'  {nombre:<49} {r["qwk"]:>8.4f} {r["recall_alta"]:>9.4f}'
          f' {r["macro_f1"]:>9.4f} {r["bal_acc"]:>8.4f}  {"✅" if cand else "❌"}')
print(f'  {"─"*118}')
for nombre, r in modelos_v9.items():
    cand = r['qwk'] >= 0.65 and r['recall_alta'] >= 0.55
    print(f'  {nombre:<49} {r["qwk"]:>8.4f} {r["recall_alta"]:>9.4f}'
          f' {r["macro_f1"]:>9.4f} {r["bal_acc"]:>8.4f}  {"✅ CANDIDATO" if cand else "❌"}')
print(f'{sep}')
print(f'  Floor H2: QWK >= 0.65 AND RecAlta >= 0.55 (congelado — Kerr 1998)')
print(f'{sep}')

all_m = {**{k: dict(qwk=v['qwk'], recall_alta=v['recall_alta']) for k,v in refs.items()},
         **{k: dict(qwk=v['qwk'], recall_alta=v['recall_alta']) for k,v in modelos_v9.items()}}
candidatos = [(k, v) for k, v in all_m.items() if v['qwk'] >= 0.65 and v['recall_alta'] >= 0.55]
mejor = max(modelos_v9.items(), key=lambda x: x[1]['qwk'])

print()
if candidatos:
    best_c = max(candidatos, key=lambda x: x[1]['qwk'])
    print('✅ H2 VERIFICADA EN VAL v9')
    print(f'   Mejor candidato: {best_c[0]}')
    print(f'   QWK={best_c[1]["qwk"]:.4f}  RecAlta={best_c[1]["recall_alta"]:.4f}')
    print()
    print('   PROTOCOLO ONE-SHOT (Kaufman et al. 2012):')
    print('   → Crear 06 evaluacion test set v9.ipynb')
    print('   → Evaluar test set UNA SOLA VEZ con el mejor candidato')
else:
    gap = 0.65 - mejor[1]['qwk']
    print(f'❌ H2 NO VERIFICADA EN VAL v9 (0 candidatos)')
    print(f'   Mejor QWK v9 val: {mejor[1]["qwk"]:.4f}  Gap: +{max(gap,0):.4f} QWK')
    print()
    print('   Acciones siguientes:')
    print('   1. D39: Threshold tuning sobre nuevo ensemble (probas mejoradas)')
    print('   2. RF con n_estimators=400 si la mejora es marginal')
    print('   3. HORIZON=1 (30 min) — revisión del horizonte de predicción')

import pickle
with open(f'{OUT_DIR}/resultados_v9_val.pkl', 'wb') as f:
    pickle.dump({'res_rf_v51': res_rf_v51, 'res_ens_v9': res_ens_v9,
                 'w_rf': W_RF, 'w_xgb': W_XGB, 'w_lgb': W_LGB}, f)
print(f'\nResultados v9 guardados: {OUT_DIR}/resultados_v9_val.pkl')



  TABLA COMPARATIVA VALIDACION — v5 / v6 / v7 / v8 / v9
  Modelo                                                 QWK   RecAlta   MacroF1   BalAcc  H2?
  ----------------------------------------------------------------------------------------------------------------------
  rf_v5                                               0.6420    0.6726    0.6560   0.6720  ❌
  xgb_v5                                              0.6232    0.6648    0.6511   0.6985  ❌
  lgb_v5                                              0.6300    0.6824    0.6533   0.7025  ❌
  ens_rfxgb_v6                                        0.6413    0.5605    0.6563   0.6164  ❌
  xgb_v7                                              0.6227    0.6445    0.6503   0.6908  ❌
  lgb_v7                                              0.6299    0.6300    0.6543   0.6832  ❌
  ens_grid_v7                                         0.6453    0.6685    0.6605   0.6779  ❌
  ens_thresh_v8                                       0.6507    0.4940    0.